# S6E5 — Colab runner

Generic notebook for running any `notebooks/0X_*.py` script on Colab Pro with GPU.

**Workflow**:
1. Mount Google Drive (Drive folder `s6e5/` is the persistent artifact store)
2. Clone GitHub repo (latest code)
3. Install pinned dependencies
4. Sync data Drive → /content (one-time copy per session — Drive IO is slow)
5. Run the target script
6. Sync artifacts /content → Drive (probs, submissions, experiments.jsonl)
7. Download submission CSV via Colab UI

**Drive folder**: https://drive.google.com/drive/folders/1N6PFShEtMj2KSYWxaQQz-6Kro1CTLilh

**Expected structure on Drive** (set up once, manually):
```
MyDrive/<your-folder>/s6e5/
├── data/
│   ├── raw/{train.csv, test.csv, sample_submission.csv}
│   └── external/f1_strategy_dataset_v4.csv
├── probs/
├── submissions/
└── experiments.jsonl   (will be created on first run)
```

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS PATH ONCE — point at the s6e5 folder in your Drive structure.
# If your Drive folder is shared, you may need to add it to 'My Drive' first or use the full path.
DRIVE_S6E5 = '/content/drive/MyDrive/kaggle/s6e5'   # ← EDIT IF DIFFERENT

!ls -la "$DRIVE_S6E5" || echo 'PATH NOT FOUND — adjust DRIVE_S6E5 above.'

## 2. Clone repo + check out latest

In [ ]:
%cd /content
!rm -rf playground-s6e5
!git clone -q https://github.com/SirGrigor/playground-s6e5.git
%cd playground-s6e5
!git log -1 --oneline

## 3. Install pinned dependencies

Uses `requirements.txt` from the repo (same pins as local — pandas 3.0.2, sklearn 1.8, xgb 3.2, lgb 4.6, catboost 1.2, pytabkit 1.7, torch 2.11 + CUDA 13).

In [ ]:
!pip install -q -r requirements.txt

# Quick sanity
import torch, lightgbm, xgboost, catboost, pytabkit, sklearn
print(f'sklearn {sklearn.__version__}  lgb {lightgbm.__version__}  xgb {xgboost.__version__}  cat {catboost.__version__}')
print(f'torch {torch.__version__}  CUDA available={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')

## 4. Sync data Drive → /content

Drive IO is slow — copy once per Colab session.

In [ ]:
import os, shutil

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/external', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)
os.makedirs('probs', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

# raw competition data
for fn in ('train.csv', 'test.csv', 'sample_submission.csv'):
    src = f'{DRIVE_S6E5}/data/raw/{fn}'
    dst = f'data/raw/{fn}'
    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f'  copied {fn}')

# external dataset
for fn in ('f1_strategy_dataset_v4.csv',):
    src = f'{DRIVE_S6E5}/data/external/{fn}'
    dst = f'data/external/{fn}'
    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f'  copied {fn}')

# sacred holdout split is in git already (data/splits/holdout_v1.parquet) — no copy needed
!ls -la data/raw data/external data/splits

## 5. Run the target script

Edit `SCRIPT` below to point at the version you want to run.

In [ ]:
SCRIPT = 'notebooks/03_v1_lgb_baseline.py'   # ← EDIT for v2, v3, etc.
EXTRA_ARGS = ''                               # e.g. '--gpu'  or  '--subsample 100000'

!python {SCRIPT} {EXTRA_ARGS}

## 6. Sync artifacts /content → Drive

Persist outputs back so they survive Colab session shutdown.

In [ ]:
import os, shutil

# Pick the version dir to sync (probs/v*_*/)
for d in sorted(os.listdir('probs')):
    src = f'probs/{d}'
    dst = f'{DRIVE_S6E5}/probs/{d}'
    os.makedirs(dst, exist_ok=True)
    for fn in os.listdir(src):
        shutil.copyfile(f'{src}/{fn}', f'{dst}/{fn}')
    print(f'  synced probs/{d}/ → Drive')

# submissions
os.makedirs(f'{DRIVE_S6E5}/submissions', exist_ok=True)
for fn in sorted(os.listdir('submissions')):
    shutil.copyfile(f'submissions/{fn}', f'{DRIVE_S6E5}/submissions/{fn}')
    print(f'  synced submissions/{fn} → Drive')

# experiments.jsonl is git-tracked too, but persist a Drive copy as backup
if os.path.exists('experiments.jsonl'):
    shutil.copyfile('experiments.jsonl', f'{DRIVE_S6E5}/experiments.jsonl')
    print('  synced experiments.jsonl → Drive')

## 7. (optional) Commit the experiment back to GitHub

Only needed if we want the JSONL append on `main`. Skip if iterating fast.

Set `GH_TOKEN` env var in Colab before running this cell (Colab UI → Secrets).

In [ ]:
# from google.colab import userdata
# token = userdata.get('GH_TOKEN')
# !git config user.email 'ilja@thefitsphere.com'
# !git config user.name 'SirGrigor (Colab)'
# !git add experiments.jsonl docs/
# !git -c 'http.extraheader=Authorization: Bearer {token}' commit -m 's6e5: append v1_lgb experiment from Colab'
# !git -c 'http.extraheader=Authorization: Bearer {token}' push https://github.com/SirGrigor/playground-s6e5.git HEAD:main
print('Commented out — uncomment after configuring GH_TOKEN secret in Colab.')

## 8. Download submission CSV

Use Colab's file browser (left sidebar) to download `submissions/v1_lgb.csv` (or the relevant version), then submit via `kaggle competitions submit` from your local machine.

In [ ]:
from google.colab import files
files.download('submissions/v1_lgb.csv')